In [2]:
import requests

def get_drug_class_pubchem(drug_name):
    # Step 1: Search for CID (Compound ID)
    search_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{drug_name}/cids/JSON"
    search_response = requests.get(search_url)
    
    if search_response.status_code != 200:
        return f"Failed to find CID for {drug_name}"

    try:
        cid = search_response.json()['IdentifierList']['CID'][0]
    except (KeyError, IndexError):
        return f"No CID found for {drug_name}"

    # Step 2: Fetch classification data from CID
    class_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/categories/compound/{cid}/JSON"
    class_response = requests.get(class_url)
    
    if class_response.status_code != 200:
        return f"Classification info not available for CID {cid}"

    try:
        categories = class_response.json()["CategoryTree"]["Categories"]
        for category in categories:
            if "Title" in category and "Pharmacology" in category["Title"]:
                for subcategory in category.get("Subcategories", []):
                    if "Title" in subcategory:
                        return subcategory["Title"]
    except Exception as e:
        return f"Error parsing classification for {drug_name}: {e}"

    return "Drug class not found"

# Example usage
drug_name = "Sertraline"
print(f"{drug_name} class: {get_drug_class_pubchem(drug_name)}")


Sertraline class: Error parsing classification for Sertraline: 'CategoryTree'
